# MedicalBot Training on Google Colab GPU
**Bước 1:** Runtime → Change runtime type → GPU (T4)

**Bước 2:** Chạy từng cell theo thứ tự

In [ ]:
# Cell 1: Kiểm tra GPU
!nvidia-smi
import torch
print('CUDA:', torch.cuda.is_available())

In [ ]:
# Cell 2: Mount Google Drive (project phải upload lên Drive trước)
from google.colab import drive
drive.mount('/content/drive')

import os
# Đổi đường dẫn nếu project ở chỗ khác
PROJECT_PATH = '/content/drive/MyDrive/MedicalBot'
os.chdir(PROJECT_PATH)
print('Working dir:', os.getcwd())
!ls

In [ ]:
# Cell 2b: Kiểm tra Python version
import sys
print('Python:', sys.version)
# Colab hiện dùng Python 3.10/3.11 — nếu thấy 3.12 thì cần đổi runtime

In [ ]:
# Cell 3: Cài dependencies
import sys
py = sys.version_info

# TF version theo Python version
if py >= (3, 12):
    # Python 3.12: dùng TF 2.16+
    !pip install -q tensorflow==2.16.1
    !pip install -q rasa==3.6.21  # version mới nhất hỗ trợ py3.12 nếu có
else:
    # Python 3.10/3.11: dùng TF 2.12
    !pip install -q 'tensorflow[and-cuda]==2.12.0'
    !pip install -q rasa==3.6.20

!pip install -q torch==2.1.0 torchvision==0.16.0 --index-url https://download.pytorch.org/whl/cu121
!pip install -q sentence-transformers==2.7.0 transformers==4.36.0 tokenizers==0.15.2 'huggingface-hub==0.23.0'
!pip install -q pandas==2.0.3 pyarrow==12.0.1 openpyxl==3.1.2 sentencepiece==0.1.99 sacremoses==0.1.1
!pip install -q 'numpy>=1.19.2,<1.24' 'networkx>=2.4,<2.7' 'regex>=2020.6,<2022.11' --force-reinstall

In [ ]:
# Cell 4: Kiểm tra TF nhận GPU
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
print('TF GPUs:', gpus)
if gpus:
    tf.config.experimental.set_memory_growth(gpus[0], True)
    print('✅ GPU sẵn sàng!')
else:
    print('❌ Không có GPU — kiểm tra Runtime type')

In [ ]:
# Cell 5: Chuẩn bị dữ liệu (bỏ qua nếu đã có data/ và health_kb.db)
import os
if not os.path.exists('knowledge_base/health_kb.db'):
    !python scripts/prepare_data.py \
        --dataset-dir dataset/ \
        --db-path knowledge_base/health_kb.db \
        --output-dir data/
else:
    print('✅ Database đã tồn tại, bỏ qua bước này')

In [ ]:
# Cell 6: Build embeddings (bỏ qua nếu đã có)
import sqlite3
conn = sqlite3.connect('knowledge_base/health_kb.db')
count = conn.execute('SELECT COUNT(*) FROM embeddings WHERE vector IS NOT NULL').fetchone()[0]
conn.close()
print(f'Embeddings đã có: {count}')

if count == 0:
    !python scripts/build_embeddings.py --db-path knowledge_base/health_kb.db
else:
    print('✅ Embeddings đã tồn tại, bỏ qua')

In [ ]:
# Cell 7: Train Rasa với GPU
import os
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'
os.environ['CUDA_VISIBLE_DEVICES'] = '0'
os.environ['SQLALCHEMY_SILENCE_UBER_WARNING'] = '1'

!rasa train --config config.yml --domain domain.yml --data data/

In [ ]:
# Cell 8: Copy model về Drive để lưu lại
import shutil, glob, os

models = sorted(glob.glob('models/*.tar.gz'))
if models:
    latest = models[-1]
    dest = f'/content/drive/MyDrive/MedicalBot/models/{os.path.basename(latest)}'
    shutil.copy(latest, dest)
    print(f'✅ Đã lưu model: {dest}')
else:
    print('❌ Không tìm thấy model')